# Step 1a Loop-1 LLM run (T4 contamination check, T5 pilot prompts)
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v3`, folder `step1a/`. Results are committed back to the same branch.

**Before running:** Runtime > Change runtime type > GPU with 24 GB or more (L4/A100) for the 32B model.

In [ ]:
#@title 1. Tokens, clone/pull wf-v3, git identity
# Needs Colab Secrets: HF_TOKEN and GH_TOKEN, notebook access on.
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
BRANCH = "wf-v3"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

In [ ]:
#@title 2. Install vLLM (a few minutes; restart the runtime if pip asks)
!pip install -q vllm

In [ ]:
#@title 3. GPU check
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import vllm, torch
print("vllm", vllm.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU: change the runtime type"

In [ ]:
#@title 4. Load the model
MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"  #@param ["Qwen/Qwen2.5-32B-Instruct-AWQ", "Qwen/Qwen2.5-14B-Instruct-AWQ"]
MAX_LEN = 32768
from vllm import LLM, SamplingParams
llm = LLM(model=MODEL, quantization="awq", max_model_len=MAX_LEN, gpu_memory_utilization=0.92)
tok = llm.get_tokenizer()

In [ ]:
#@title 5. Exact prompt token counts (stops if any prompt > 30000)
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from pathlib import Path
from s1a import llm_cache as lc
O = Path("results/loop1_check"); PT = O / "pilot_train"
VARIANTS = ["full", "nocontext", "nohints"]
def chat_prompt(text):
    return tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)
prompts = {v: (PT / "prompts" / f"{v}.txt").read_text(encoding="utf-8") for v in VARIANTS}
counts = {v: len(tok(chat_prompt(p), add_special_tokens=False).input_ids) for v, p in prompts.items()}
print(counts)
too_long = {v: n for v, n in counts.items() if n > 30000}
assert not too_long, f"prompt over 30000 tokens: {too_long}; shorten it (W3) before running"

In [ ]:
#@title 6. Generation helper (temperature 0, JSON-guided for T5)
# Guided JSON: vLLM >= 0.11 uses StructuredOutputsParams(structured_outputs=...), older uses GuidedDecodingParams(guided_decoding=...).
def sampling(max_tokens, guided):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if guided:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=lc.ARRAY_SCHEMA)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=lc.ARRAY_SCHEMA)
    return SamplingParams(**kw)

def generate(text, max_tokens, guided):
    return llm.generate([chat_prompt(text)], sampling(max_tokens, guided))[0].outputs[0].text

In [ ]:
#@title 7. T4: the two questions, NO dialogue data
questions = [q for q in (O / "t4_questions.txt").read_text(encoding="utf-8").splitlines() if q.strip()]
assert len(questions) == 2
params = {"temperature": 0, "max_tokens": 1024, "guided_json": False}
out = O / "contamination_raw.jsonl"
for q in questions:
    k = lc.key(MODEL, params, q, "t4")
    if lc.get(out, k) is None:
        lc.put(out, k, {"question": q, "model": MODEL, "params": params, "run_tag": "t4", "raw": generate(q, 1024, False)})
    print(q, "\n ->", lc.get(out, k)["raw"][:600], "\n")

In [ ]:
#@title 8. T5: full/r1, full/r2, nocontext/r1, nohints/r1 (cached by model+params+prompt+run_tag)
import hashlib
RUNS = [("full", "r1"), ("full", "r2"), ("nocontext", "r1"), ("nohints", "r1")]
for variant, tag in RUNS:
    prompt = prompts[variant]
    max_tokens = min(12000, MAX_LEN - counts[variant] - 64)
    params = {"temperature": 0, "max_tokens": max_tokens, "guided_json": "loop1_array_schema_v1"}
    k = lc.key(MODEL, params, prompt, tag)
    path = PT / f"responses_{variant}_{tag}.jsonl"
    row = lc.get(path, k)
    if row is None:
        raw = generate(prompt, max_tokens, True)
        parsed, errors = lc.parse_response(raw)
        row = {"prompt_sha": hashlib.sha256(prompt.encode("utf-8")).hexdigest(), "model": MODEL, "params": params,
               "run_tag": tag, "raw": raw, "parsed": parsed, "errors": errors}
        lc.put(path, k, row)
    print(variant, tag, "objects:", len(row["parsed"]), "errors:", len(row["errors"]), row["errors"][:3])

In [ ]:
#@title 9. Commit and push results/loop1_check to wf-v3
!bash colab/sync.sh push-results "T4 + T5 responses"